# Data Cleaning

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')
from src.helpers import *

## Loading Datasets

In [2]:
shootings_df = pd.read_csv('../Data/Shootings_(2006-Present)_20260401.csv')              # loading three dataset files
offenders_df = pd.read_csv('../Data/Shooting_Offenders_(2006-Present)_20260401.csv')
victims_df = pd.read_csv('../Data/Shooting_Victims_(2006-Present)_20260401.csv')

In [3]:
combined_df = pd.merge(shootings_df, offenders_df, on='INCIDENT_KEY', how='outer')             # merging the datasets to a combined dataset
combined_df = pd.merge(combined_df, victims_df, on='INCIDENT_KEY', how='outer')

## Cleaning Shootings Dataset

In [4]:
shootings_df = title_case(shootings_df)
shootings_df = datetimecols(shootings_df, columns=['OCCUR_DATE', 'OCCUR_TIME'], column_name= 'OCCUR_DATETIME')
shootings_df = fill_nan(shootings_df)
shootings_df = shootings_df.drop(columns=['Latitude', 'Longitude'], axis=1)
shootings_df.to_csv('../Cleaned_Data/Cleaned_Shootings_(2006-Present)_20260401.csv', index=False)

- Used `title_case` to convert every string object format to title case
- Used `datetimecols` to combine date and time into one column and remove the original columns
- Used `fill_nan` to fill the `NaN` values for all string object columns
- Dropped `Latitude` and `Longitude` as it's reduntant columns, because `X_COORD_CD` and `Y_COORD_CD`, also represent coordinates. Moreover, `Longitude` and `Latitude` had more missing values compare to coordinates in city coordinates system type.

## Cleaning Victims Dataset

In [5]:
age_list = ['1022']
victims_df = title_case(victims_df)
victims_df = fill_nan(victims_df)
victims_df['VICTIM_AGE_GROUP'] = victims_df['VICTIM_AGE_GROUP'].replace(age_list, 'Unknown')
victims_df.to_csv('../Cleaned_Data/Cleaned_Shooting_Victims_(2006-Present)_20260401.csv', index=False)

- Used `title_case` to convert every string object format to title case
- Used `fill_nan` to fill the `NaN` values for all string object columns

## Cleaning Offenders Dataset

In [6]:
age_list = ['224', '940', '1020', '1822', '1028', '2021']
offenders_df = title_case(offenders_df)
offenders_df = fill_nan(offenders_df)
offenders_df['PERP_AGE_GROUP'] = offenders_df['PERP_AGE_GROUP'].replace(age_list, 'Unknown')
offenders_df.to_csv('../Cleaned_Data/Cleaned_Shooting_Offenders_(2006-Present)_20260401.csv', index=False)

- Used `title_case` to convert every string object format to title case
- Used `fill_nan` to fill the `NaN` values for all string object columns

## Cleaning Combined Dataset

In [7]:
combined_df = pd.merge(shootings_df, offenders_df, on='INCIDENT_KEY', how='outer')
combined_df = pd.merge(combined_df, victims_df, on='INCIDENT_KEY', how='outer')
combined_df = fill_nan(combined_df)
combined_df = auto_removal(combined_df, columns_check=['PERP_ID', 'VICTIM_ID', 'BORO'])
combined_df.to_csv('../Cleaned_Data/Combined_Cleaned_Shooting_Incidents_(2006-Present)_20260401.csv', index=False)

Merging the cleaned dataframes for combined dataset, and using `fill_nan` again, to cover up the `NaN` resulting from a outer join, as many victims records don't have offender records. And using `auto_removal` to remove rows which have `Unknown` for `PERP_ID`, `VICTIM_ID` and `BORO`, because absence of these three important values, makes the records useless, as it isn't possible to get useful insights from those records, without any offender records, victim records and part of the city. And absence of these three almost indicates a dummy entry by mistake. 